# Sesión 1 · Notebook previo
## Cómo funciona un LLM y cómo usarlo bien

**Curso:** Ciencia de datos aumentada con IA: del dato a la decisión · Maestría en Ciencia de Datos
**Tiempo estimado:** 40 a 50 minutos · **Entrega:** este notebook ejecutado, antes de la clase

### Qué vas a hacer
1. Configurar el entorno y hacer tu primera llamada a un LLM por API.
2. Ver cómo un texto se convierte en tokens y estimar el costo de una consulta.
3. Comparar **dos versiones de un prompt** de extracción a JSON y medir cuál funciona mejor sobre 10 tickets etiquetados.
4. (Opcional) Repetir la medición con un modelo pequeño local en Ollama.

### Qué se espera
Las celdas marcadas con **TU TURNO** requieren que escribas o respondas algo. Todo lo demás está listo para ejecutarse. Recuerda declarar tu nivel de uso de IAG al final.

## 0. Configuración

In [ ]:
!pip install -q tiktoken anthropic google-genai pandas requests

In [ ]:
import os, json, re, time, unicodedata
import pandas as pd

# --- Elige UN proveedor y pega su clave en los "Secretos" de Colab (icono de llave, panel izquierdo) ---
# Anthropic:  secreto ANTHROPIC_API_KEY   |   Google AI Studio (Gemini):  secreto GEMINI_API_KEY
PROVEEDOR = "gemini"          # "anthropic" o "gemini"

# Nombres de modelo por defecto. Si el proveedor los retiró, reemplázalos por un modelo económico vigente
# (consulta la documentación del proveedor).
MODELOS = {"anthropic": "claude-haiku-4-5", "gemini": "gemini-3.5-flash-lite"}
MODELO = MODELOS[PROVEEDOR]

def obtener_clave(nombre):
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        return os.environ.get(nombre) or input(f"Pega tu {nombre}: ")

if PROVEEDOR == "anthropic":
    import anthropic
    _cliente = anthropic.Anthropic(api_key=obtener_clave("ANTHROPIC_API_KEY"))
else:
    from google import genai
    _cliente = genai.Client(api_key=obtener_clave("GEMINI_API_KEY"))

def llm_api(prompt, temperatura=0.0, max_tokens=400):
    """Devuelve (texto, tokens_entrada, tokens_salida)."""
    if PROVEEDOR == "anthropic":
        r = _cliente.messages.create(model=MODELO, max_tokens=max_tokens, temperature=temperatura,
                                     messages=[{"role": "user", "content": prompt}])
        return r.content[0].text, r.usage.input_tokens, r.usage.output_tokens
    r = _cliente.models.generate_content(model=MODELO, contents=prompt,
                                         config={"temperature": temperatura, "max_output_tokens": max_tokens})
    u = r.usage_metadata
    return r.text, u.prompt_token_count, u.candidates_token_count

texto, t_in, t_out = llm_api("Explica en una frase qué es un token en un modelo de lenguaje.")
print(texto); print(f"\nTokens de entrada: {t_in} | tokens de salida: {t_out}")

## 1. Del texto a los tokens
Un LLM no lee palabras: lee **tokens** (fragmentos de texto). Esto afecta el costo, el límite de contexto y hasta lo bien que el modelo maneja ciertos idiomas o números.
Usamos `tiktoken` solo para **visualizar** cómo se parte un texto; cada modelo tiene su propio tokenizador, así que los conteos reales por API pueden diferir un poco.

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

def ver_tokens(texto):
    ids = enc.encode(texto)
    piezas = [enc.decode([i]) for i in ids]
    print(f"{len(ids)} tokens para {len(texto)} caracteres  ->  {piezas}")

ver_tokens("La inteligencia artificial generativa transforma la ciencia de datos.")
ver_tokens("Generative artificial intelligence is transforming data science.")
ver_tokens("El valor total fue $2.450.000 el 15/03/2026.")
ver_tokens("df.groupby('categoria')['monto'].sum().reset_index()")

### TU TURNO 1
Prueba con **tres textos propios** (por ejemplo, una frase técnica en español, una en inglés y una con números o código) y responde en la celda de abajo:
1. ¿Qué idioma o tipo de contenido necesita más tokens por carácter?
2. ¿Por qué importa esto para el costo y para la ventana de contexto?

In [ ]:
# Escribe aquí tus tres textos
ver_tokens("...")
ver_tokens("...")
ver_tokens("...")

respuesta_1 = """
Escribe aquí tu respuesta a las dos preguntas.
"""

### Costo de una consulta
El costo de una llamada se calcula con los tokens de entrada y de salida y los precios por millón de tokens del proveedor.
**Busca los precios vigentes de tu modelo en la página de precios del proveedor** y escríbelos abajo (en dólares por millón de tokens).

In [ ]:
PRECIO_ENTRADA_POR_MILLON = 0.0   # TU TURNO: reemplaza por el precio real (USD por millón de tokens de entrada)
PRECIO_SALIDA_POR_MILLON  = 0.0   # TU TURNO: reemplaza por el precio real (USD por millón de tokens de salida)

def costo_usd(t_in, t_out):
    return (t_in * PRECIO_ENTRADA_POR_MILLON + t_out * PRECIO_SALIDA_POR_MILLON) / 1_000_000

print(f"Costo de la llamada anterior: USD {costo_usd(t_in, t_out):.6f}")
print(f"Costo estimado de 10.000 llamadas iguales: USD {costo_usd(t_in, t_out) * 10_000:.2f}")

## 2. Temperatura: qué cambia cuando muestreas distinto
El modelo asigna probabilidades a los siguientes tokens y luego **muestrea**. Una temperatura baja hace la salida más determinista; una alta, más variable. Ejecuta la misma pregunta tres veces con cada temperatura.

In [ ]:
pregunta = "Dame un nombre creativo para un proyecto de análisis de datos de ventas. Responde solo con el nombre."
for temp in (0.0, 1.0):
    print(f"--- temperatura {temp} ---")
    for _ in range(3):
        print(llm_api(pregunta, temperatura=temp, max_tokens=30)[0].strip())

### TU TURNO 2
¿Qué temperatura usarías para **extraer datos de un documento** y cuál para **generar ideas**? Justifica en dos líneas.

In [9]:
respuesta_2 = """
Escribe aquí tu respuesta.
"""

## 3. Extracción estructurada: dos prompts, una métrica
Vas a extraer información de **tickets de soporte** y devolverla como JSON. Un prompt vago puede "funcionar a ojo", pero solo la medición te dice si mejoró.

Cada ticket tiene su respuesta correcta (etiqueta) para calcular la **exactitud por campo**:
`producto`, `categoria` (facturacion, envio, calidad, soporte_tecnico, otro), `urgencia` (alta, media, baja), `sentimiento` (negativo, neutro, positivo) y `monto` (número en pesos o `null`).

In [ ]:
TICKETS = [
 ("Compré el portátil Lenovo IdeaPad hace dos semanas y aún no me llega. Pagué $2.450.000 y nadie me da razón del envío. Estoy muy molesto.",
  {"producto":"portátil Lenovo IdeaPad","categoria":"envio","urgencia":"alta","sentimiento":"negativo","monto":2450000}),
 ("Me cobraron dos veces la suscripción mensual de Spotify Premium este mes, $16.900 cada una. Necesito que me devuelvan uno de los cobros.",
  {"producto":"Spotify Premium","categoria":"facturacion","urgencia":"media","sentimiento":"negativo","monto":16900}),
 ("Excelente servicio, el técnico resolvió el problema del router en menos de una hora. Muchas gracias.",
  {"producto":"router","categoria":"soporte_tecnico","urgencia":"baja","sentimiento":"positivo","monto":None}),
 ("La aplicación del banco se cierra sola cada vez que intento transferir dinero desde mi celular Samsung. Es urgente, tengo pagos pendientes hoy.",
  {"producto":"aplicación del banco","categoria":"soporte_tecnico","urgencia":"alta","sentimiento":"negativo","monto":None}),
 ("Quisiera saber si la lavadora LG WM3900 tiene garantía de dos años. La compré por $1.890.000.",
  {"producto":"lavadora LG WM3900","categoria":"otro","urgencia":"baja","sentimiento":"neutro","monto":1890000}),
 ("Los audífonos Sony WH-1000XM5 llegaron con el cable roto. Quiero cambio o reembolso de los $1.150.000.",
  {"producto":"audífonos Sony WH-1000XM5","categoria":"calidad","urgencia":"media","sentimiento":"negativo","monto":1150000}),
 ("Mi factura de internet llegó por $189.000, pero mi plan cuesta $99.000. Por favor revisen el cobro.",
  {"producto":"internet","categoria":"facturacion","urgencia":"media","sentimiento":"negativo","monto":189000}),
 ("Hola, el televisor Samsung QLED de 55 pulgadas se dañó a los tres meses. Ya escribí tres veces y no responden. Me siento ignorado.",
  {"producto":"televisor Samsung QLED 55 pulgadas","categoria":"calidad","urgencia":"alta","sentimiento":"negativo","monto":None}),
 ("Gracias por la rapidez con el envío de mi pedido de zapatos Nike Air Max, llegaron un día antes. Costaron $520.000.",
  {"producto":"zapatos Nike Air Max","categoria":"envio","urgencia":"baja","sentimiento":"positivo","monto":520000}),
 ("Necesito cambiar la dirección de entrega de mi nevecón Haceb antes de que salga hoy del centro de distribución.",
  {"producto":"nevecón Haceb","categoria":"envio","urgencia":"alta","sentimiento":"neutro","monto":None}),
]
CAMPOS = ["producto", "categoria", "urgencia", "sentimiento", "monto"]
print(f"{len(TICKETS)} tickets cargados")

### Prompt 1 (versión inicial) y Prompt 2 (versión diseñada)
El **Prompt 1** es el que escribiría cualquiera a la primera. El **Prompt 2** define el esquema, los valores permitidos, la regla para el monto, da dos ejemplos (few-shot) y exige responder solo con JSON.

In [11]:
PROMPT_V1 = """Extrae del siguiente ticket de soporte el producto, la categoría, la urgencia, el sentimiento y el monto, y devuélvelo como JSON.

Ticket: {texto}"""

PROMPT_V2 = """Eres un asistente que extrae información de tickets de soporte de una tienda en Colombia.
Devuelve ÚNICAMENTE un objeto JSON válido, sin texto adicional ni comillas de código, con exactamente estas claves:
- "producto": nombre del producto o servicio mencionado (texto corto).
- "categoria": una de ["facturacion", "envio", "calidad", "soporte_tecnico", "otro"].
- "urgencia": una de ["alta", "media", "baja"]. Es "alta" si el cliente indica que es urgente, tiene una fecha límite o ya insistió varias veces.
- "sentimiento": una de ["negativo", "neutro", "positivo"].
- "monto": número entero en pesos colombianos que el cliente dice haber pagado o que le cobraron (sin símbolos ni puntos). Si no se menciona ningún valor, usa null.

Ejemplo 1
Ticket: El monitor Dell U2723 llegó con una línea en la pantalla. Lo pagué en $1.300.000 y quiero que lo cambien.
JSON: {{"producto": "monitor Dell U2723", "categoria": "calidad", "urgencia": "media", "sentimiento": "negativo", "monto": 1300000}}

Ejemplo 2
Ticket: ¿Me pueden confirmar el horario de atención de la tienda en Cali?
JSON: {{"producto": "tienda", "categoria": "otro", "urgencia": "baja", "sentimiento": "neutro", "monto": null}}

Ticket: {texto}
JSON:"""

### Funciones de evaluación
La métrica es la **exactitud por campo** (cuántos tickets acierta el modelo en cada campo), la **tasa de JSON válido**, los tokens usados, la latencia y el costo estimado.
Para `producto` se acepta coincidencia flexible (ignora tildes y mayúsculas, y acepta que uno contenga al otro); para `monto` se compara el número.

In [12]:
def normalizar(s):
    s = unicodedata.normalize("NFD", str(s).lower())
    return "".join(c for c in s if unicodedata.category(c) != "Mn").strip()

def a_numero(v):
    if v is None: return None
    if isinstance(v, (int, float)): return int(v)
    limpio = re.sub(r"[^\d]", "", str(v))
    return int(limpio) if limpio else None

def parsear_json(texto):
    m = re.search(r"\{.*\}", texto, flags=re.S)
    if not m: return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def campo_correcto(campo, pred, real):
    if pred is None: return False
    if campo == "monto":
        return a_numero(pred.get("monto")) == real["monto"]
    p, r = normalizar(pred.get(campo, "")), normalizar(real[campo])
    return (r in p or p in r) and p != "" if campo == "producto" else p == r

RESULTADOS = {}

def evaluar(nombre, fn_llm, plantilla, pausa=0.0):
    filas, t_in_tot, t_out_tot, lat = [], 0, 0, []
    for texto, real in TICKETS:
        t0 = time.time()
        crudo, t_in, t_out = fn_llm(plantilla.format(texto=texto))
        lat.append(time.time() - t0); t_in_tot += t_in; t_out_tot += t_out
        pred = parsear_json(crudo)
        fila = {"ticket": texto[:50] + "...", "json_valido": pred is not None}
        for c in CAMPOS: fila[c] = campo_correcto(c, pred, real)
        fila["salida"] = crudo.strip().replace("\n", " ")[:120]
        filas.append(fila); time.sleep(pausa)
    df = pd.DataFrame(filas)
    resumen = {"json_valido": df["json_valido"].mean(), **{c: df[c].mean() for c in CAMPOS}}
    resumen["exactitud_media"] = sum(resumen[c] for c in CAMPOS) / len(CAMPOS)
    resumen.update({"tokens_entrada": t_in_tot, "tokens_salida": t_out_tot,
                    "latencia_media_s": sum(lat) / len(lat), "costo_usd": costo_usd(t_in_tot, t_out_tot)})
    RESULTADOS[nombre] = resumen
    return df, resumen

In [ ]:
df_v1, res_v1 = evaluar("API · Prompt 1", llm_api, PROMPT_V1)
df_v1

In [ ]:
# Añadimos un parámetro 'pausa' de 4 segundos entre llamadas para evitar el error de cuota (429)
df_v2, res_v2 = evaluar("API · Prompt 2", llm_api, PROMPT_V2, pausa=4.0)
df_v2

In [ ]:
pd.DataFrame(RESULTADOS).T.round(3)

### TU TURNO 3
Observa los tickets donde cada prompt falla (`False`) y lee la columna `salida`.
1. ¿Qué campos mejoraron con el Prompt 2 y por qué crees que fue así?
2. ¿Cuánto más costó el Prompt 2 en tokens? ¿Vale la pena?
3. **Modifica el Prompt 2** (por ejemplo, agrega un ejemplo más o cambia una definición), llámalo `PROMPT_V3`, evalúalo en la celda siguiente y compara.

In [ ]:
PROMPT_V3 = PROMPT_V2   # TU TURNO: modifica esta versión
df_v3, res_v3 = evaluar("API · Prompt 3 (mío)", llm_api, PROMPT_V3)

respuesta_3 = """
Escribe aquí tus respuestas a las tres preguntas.
"""
pd.DataFrame(RESULTADOS).T.round(3)

## 4. (Opcional) Lo mismo con un modelo pequeño local en Ollama
Si corres este notebook **en tu computador** (Jupyter o VS Code) con [Ollama](https://ollama.com) instalado, puedes medir cuánto pierdes o ganas al usar un modelo local. En Colab esta parte no funciona porque el servidor de Ollama corre en tu máquina, así que puedes saltarla.

Pasos: instala Ollama, descarga un modelo pequeño (por ejemplo `ollama pull llama3.2`) y comprueba que el servicio esté activo.

In [ ]:
import requests
MODELO_OLLAMA = "llama3.2"   # cambia por el modelo que hayas descargado

def llm_ollama(prompt, temperatura=0.0):
    r = requests.post("http://localhost:11434/api/generate", timeout=300,
                      json={"model": MODELO_OLLAMA, "prompt": prompt, "stream": False,
                            "options": {"temperature": temperatura}})
    r.raise_for_status(); d = r.json()
    return d["response"], d.get("prompt_eval_count", 0), d.get("eval_count", 0)

try:
    requests.get("http://localhost:11434", timeout=2)
    OLLAMA_ACTIVO = True
except Exception:
    OLLAMA_ACTIVO = False
print("Ollama activo" if OLLAMA_ACTIVO else "Ollama no detectado: puedes saltar esta sección.")

In [ ]:
if OLLAMA_ACTIVO:
    evaluar(f"Ollama ({MODELO_OLLAMA}) · Prompt 2", llm_ollama, PROMPT_V2)
pd.DataFrame(RESULTADOS).T.round(3)

## 5. Cierre
### TU TURNO 4 · Reflexión final (máx. 5 líneas)
¿Qué aprendiste sobre tokens, temperatura y diseño de prompts que te sirva para tu proyecto? Si hiciste la parte de Ollama, ¿cómo se compara con la API en calidad, latencia y costo?

In [22]:
respuesta_4 = """
Escribe aquí tu reflexión.
"""

# Declaración de uso de IAG (obligatoria)
DECLARACION_IAG = """
Nivel de uso de IAG: (1 a 5, según el acuerdo del curso)
Herramientas usadas y para qué:
Cómo verifiqué la calidad y veracidad del contenido generado:
"""

### Antes de entregar
- Ejecuta **Entorno de ejecución → Reiniciar y ejecutar todo** y verifica que no haya errores.
- Comprueba que las celdas `respuesta_1` a `respuesta_4` y `DECLARACION_IAG` estén completas.
- **No compartas tu clave de API.** Borra cualquier clave que hayas pegado en una celda antes de compartir el notebook.
- Guarda una copia en Drive y entrega el enlace según indique el profesor.